In [1]:
pip install dagshub mlflow scikit-learn xgboost skops --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 kB 6.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 69.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 16.5 MB/s eta 0:00:0000:010:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

## Imports

In [2]:
import mlflow
import mlflow.sklearn
import dagshub
import skops.io as sio
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import Ridge, Lasso
from sklearn.feature_selection import RFE

## DagsHub/MLflow Initialization

In [3]:
dagshub.init(repo_owner='adzid23', repo_name='Freeuni_ML_House_Prices', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d0a94811-a307-43ba-bbb2-455fb0aeb4df&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=d57f4793ada877eb8630ff0800e201c51852865b94e1b1772be6387952a07492




Accessing as adzid23

Initialized MLflow to track repo "adzid23/Freeuni_ML_House_Prices"

Repository adzid23/Freeuni_ML_House_Prices initialized!

## Custom Transformer Definitions

In [4]:
class HouseAgeAdder(BaseEstimator, TransformerMixin):
    """Adds HouseAge and HouseRemodelAge from year columns."""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X['HouseAge']        = X['YrSold'] - X['YearBuilt']
        X['HouseRemodelAge'] = X['YrSold'] - X['YearRemodAdd']
        return X

class AreaAdder(BaseEstimator, TransformerMixin):
    """Combines area columns into aggregate features."""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X['TotalSF']      = X['1stFlrSF'] + X['2ndFlrSF'] + X['BsmtFinSF1'] + X['BsmtFinSF2']
        X['TotalArea']    = X['GrLivArea'] + X['TotalBsmtSF']
        X['TotalPorchSF'] = (X['OpenPorchSF'] + X['3SsnPorch'] +
                             X['EnclosedPorch'] + X['ScreenPorch'] + X['WoodDeckSF'])
        return X

class BathAdder(BaseEstimator, TransformerMixin):
    """Combines bathroom columns into a single TotalBaths feature."""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X['TotalBaths'] = (X['FullBath'] + 0.5 * X['HalfBath'] +
                           X['BsmtFullBath'] + 0.5 * X['BsmtHalfBath'])
        return X

class ColumnNameRestorer(BaseEstimator, TransformerMixin):
    """Restores column names lost after ColumnTransformer."""
    def __init__(self, column_transformer):
        self.column_transformer = column_transformer
    def fit(self, X, y=None):
        self.feature_names_ = self.column_transformer.get_feature_names_out()
        return self
    def transform(self, X):
        return pd.DataFrame(X, columns=self.feature_names_)

class CorrelationFilter(BaseEstimator, TransformerMixin):
    """Drops one feature from each highly-correlated pair."""
    def __init__(self, threshold=0.90):
        self.threshold = threshold
        self.features_to_drop_ = None
        self.selected_features_ = None
    def fit(self, X, y=None):
        X = pd.DataFrame(X).reset_index(drop=True)
        corr_matrix = X.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.features_to_drop_ = set()
        for col in upper.columns:
            if any(upper[col] > self.threshold):
                partners = upper.index[upper[col] > self.threshold].tolist()
                for partner in partners:
                    if corr_matrix[col].mean() >= corr_matrix[partner].mean():
                        self.features_to_drop_.add(col)
                    else:
                        self.features_to_drop_.add(partner)
        self.selected_features_ = [
            col for col in X.columns if col not in self.features_to_drop_
        ]
        return self
    def transform(self, X):
        X = pd.DataFrame(X).reset_index(drop=True)
        return X[self.selected_features_]

class RFESelector(BaseEstimator, TransformerMixin):
    """Wraps sklearn RFE with a Ridge estimator for feature ranking."""
    def __init__(self, n_features_to_select=60):
        self.n_features_to_select = n_features_to_select
        self.selected_features_ = None
        self.rfe_ = None
    def fit(self, X, y):
        X = pd.DataFrame(X).reset_index(drop=True)
        self.feature_names_ = list(X.columns)
        estimator = Ridge(alpha=1.0)
        self.rfe_ = RFE(estimator=estimator, n_features_to_select=self.n_features_to_select)
        self.rfe_.fit(X, y)
        self.selected_features_ = [
            f for f, s in zip(self.feature_names_, self.rfe_.support_) if s
        ]
        return self
    def transform(self, X):
        X = pd.DataFrame(X).reset_index(drop=True)
        X.columns = self.feature_names_
        return X[self.selected_features_]

## Load Preprocessing Pipeline from MLflow

In [5]:
PREPROCESSING_RUN_ID = "f06e53f2cc404745bdb505dd56f28536"

local_path = mlflow.artifacts.download_artifacts(
    run_id=PREPROCESSING_RUN_ID,
    artifact_path="preprocessing_pipeline.skops"
)
trusted_types = sio.get_untrusted_types(file=local_path)
preprocessing_pipeline = sio.load(local_path, trusted=trusted_types)
print("Preprocessing pipeline loaded")

Preprocessing pipeline loaded


## Load Champion Model from Registry

In [6]:
best_model = mlflow.sklearn.load_model("models:/HousePriceBestModel@Champion")
print(f"Model loaded: {type(best_model).__name__}")

Model loaded: VotingRegressor


## Inference & Generate Submission

In [7]:
test_df = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

X_test_transformed = preprocessing_pipeline.transform(test_df)
print(f"X_test_transformed: {X_test_transformed.shape}")

sale_price = np.expm1(best_model.predict(X_test_transformed.to_numpy()))

pd.DataFrame({"Id": test_df["Id"], "SalePrice": sale_price}) \
  .to_csv("/kaggle/working/submission.csv", index=False)
print("submission.csv saved")

X_test_transformed: (1459, 60)
submission.csv saved


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but Lasso was fitted with feature names
  warnings.warn(
